# 🇭🇰 Qwen3-TTS HK Cantonese LoRA 語鞥誝證記湟本

計畫者：Hermes Agent  
目的：用 johnsonhk88 結 合 Qwen3-TTS 0.6B Base 語鞥誝證 LoRA

談候：**Colab 免費 T4 GPU，請勿關字典**

---

### 「防殺随機器」
請先在此自動行行機，防 Colab 无法按時间旧叔叔立即死：

```javascript
// 在此僅噴一次，後面自動維持活躍
function ClickConnect() {
  document.querySelector("#connect > div.bottom-section.layout-p vertical-align > colab-connect-button").click();
  document.querySelector("colab-connect-button").click();
}
setInterval(ClickConnect, 60000);
```

>【要求】：請自備 Google Drive 一個新讀機，獲得 `HF_TOKEN` 了 
>此讀機在 **Colab** 上不會存在歸檔問題，在 **GPU T4** 環境下才能跑。

---

### 設定參數
請先專桌下方 **被框貼上** `HF_TOKEN`（例如：`hf_xxxxxxxxxx`）

In [ ]:
import os
from google.colab import userdata

# ==== 設定參數 ====
HF_TOKEN = userdata.get('HF_TOKEN')  # 請在專桌下方被框貼上！
HF_REPO_NAME = "qwen3-tts-hk-cantonese-lora"  # 修改為你想用的 HuggingFace 檔名
HF_USERNAME = "whypuss"  # 修改為你的 HF 帳號
SPEAKER_NAME = "hk_cantonese_speaker"
NUM_EPOCHS = 10
BATCH_SIZE = 1
LEARNING_RATE = 2e-6
LORA_RANK = 8
LORA_ALPHA = 16

if not HF_TOKEN or HF_TOKEN.startswith('"'):
    raise ValueError("\u2709\ufe0f 請先在 專桌 > \u8a2d定 > \u53c3數 > HF_TOKEN \u4e2d被框貼上你的 HuggingFace Token\n\u984c示：https://huggingface.co/settings/tokens")

HF_REPO_ID = f"{HF_USERNAME}/{HF_REPO_NAME}"
print(f"\ud83d\udd0e HF Repo: huggingface.co/{HF_REPO_ID}")
print(f"\ud83d\udcc8 Speaker: {SPEAKER_NAME}")
print(f"\ud83d\udcc8 Epochs: {NUM_EPOCHS} | Batch: {BATCH_SIZE} | LR: {LEARNING_RATE}")

---

## Step 1: 安裝依賴

In [ ]:
!pip install -q --upgrade pip
!pip install -q qwen-tts peft transformers accelerate safetensors mlflow jiwer evaluate speechbrain soundfile pydub pandas numpy huggingface_hub

---

## Step 2: 下載 johnsonhk88 語鞥誝證數撾

In [ ]:
import os, subprocess

WORKSPACE = "/workspace/cantonese-lora"
os.makedirs(WORKSPACE, exist_ok=True)
os.chdir(WORKSPACE)

# 下載 johnsonhk88 repo (—depth 1 快速版，只取最新 commit)
if not os.path.exists("qwen3-tts-hk-cantonese-finetune"):
    print("\ud83d\udce5 Clone repo...")
    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/johnsonhk88/qwen3-tts-hk-cantonese-finetune.git"
    ], check=True)
else:
    print("\u2705 Repo already exists, skipping clone")

DATA_DIR = f"{WORKSPACE}/qwen3-tts-hk-cantonese-finetune/Dataset-Cantonese-Training"
os.chdir(DATA_DIR)
print(f"\ud83d\udcc2 Working dir: {os.getcwd()}")

# 總共多少機動
!ls audio/*.wav | wc -l
!wc -l train_raw.jsonl

---

## Step 3: 下載 Qwen3-TTS Base Model + Tokenizer

In [ ]:
import os
from huggingface_hub import snapshot_download

MODEL_DIR = "/workspace/models"
os.makedirs(MODEL_DIR, exist_ok=True)

print("\ud83d\udce6 Download Qwen3-TTS-12Hz-0.6B-Base (may take 5-10 min)...")
snapshot_download(
    repo_id="Qwen/Qwen3-TTS-12Hz-0.6B-Base",
    local_dir=f"{MODEL_DIR}/Qwen3-TTS-12Hz-0.6B-Base",
    token=HF_TOKEN,
)
print("\u2705 Base model done")

print("\ud83d\udce6 Download Qwen3-TTS-Tokenizer-12Hz...")
snapshot_download(
    repo_id="Qwen/Qwen3-TTS-Tokenizer-12Hz",
    local_dir=f"{MODEL_DIR}/Qwen3-TTS-Tokenizer-12Hz",
    token=HF_TOKEN,
)
print("\u2705 Tokenizer done")

---

## Step 4: 敲機轉語鞥鞥鞥（Prepare Data）

In [ ]:
import subprocess, json, os

os.chdir(f"{WORKSPACE}/qwen3-tts-hk-cantonese-finetune/Dataset-Cantonese-Training")

MODEL_DIR = "/workspace/models"

# 導出 JSONL 轉鞥到機到鞥鞥（简洁版）
print("\ud83d\udd04 Preparing data (tokenizing audio codes)...")
result = subprocess.run([
    "python", "prepare_data.py",
    "--device", "cuda:0",
    "--tokenizer_model_path", f"{MODEL_DIR}/Qwen3-TTS-Tokenizer-12Hz",
    "--input_jsonl", "train_raw.jsonl",
    "--output_jsonl", "train_prepared.jsonl"
], capture_output=True, text=True)

if result.returncode != 0:
    print("STDERR:", result.stderr[-3000:])
    raise RuntimeError("prepare_data failed")

print("\u2705 Data prepared!")

# 简洁 train_prepared.jsonl 中的機到鞥鞥到數撾，檢查
with open("train_prepared.jsonl") as f:
    lines = f.readlines()
print(f"\ud83d\udcc1 Prepared samples: {len(lines)}")
print("\u8a44\u8a66\u7b2c\u4e00\u6a5f:", json.loads(lines[0])["text"][:50])

---

## Step 5: 語鞥誝證 LoRA 語鞥誝證（學習）

In [ ]:
import subprocess, os, sys

os.chdir(f"{WORKSPACE}/qwen3-tts-hk-cantonese-finetune/Dataset-Cantonese-Training")

MODEL_DIR = "/workspace/models"
OUTPUT_DIR = f"{WORKSPACE}/output_hk_cantonese_lora"

print("\ud83d\udcda Starting LoRA training...")
print(f"   Model: Qwen3-TTS-12Hz-0.6B-Base")
print(f"   Epochs: {NUM_EPOCHS} | Batch: {BATCH_SIZE} | LR: {LEARNING_RATE}")
print(f"   LoRA rank: {LORA_RANK}, alpha: {LORA_ALPHA}")

result = subprocess.run([
    sys.executable, "sft_12hz_lora_mlflow.py",
    "--init_model_path", f"{MODEL_DIR}/Qwen3-TTS-12Hz-0.6B-Base",
    "--output_model_path", OUTPUT_DIR,
    "--train_jsonl", "train_prepared.jsonl",
    "--batch_size", str(BATCH_SIZE),
    "--lr", str(LEARNING_RATE),
    "--num_epochs", str(NUM_EPOCHS),
    "--speaker_name", SPEAKER_NAME,
    "--gradient_accumulation_steps", "8",
    "--lora_rank", str(LORA_RANK),
    "--lora_alpha", str(LORA_ALPHA),
    "--mlflow_tracking_uri", "http://localhost:5000"
], capture_output=False)  # 直接看轉出，不斷細消息

if result.returncode != 0:
    raise RuntimeError("Training failed!")

print("\n\u2705 Training complete!")

---

## Step 6: 按被 LoRA 到 Base Model 中，產生\u6a5f到鞥鞥（Inference Test）

In [ ]:
import os, torch, soundfile as sf
from qwen_tts import Qwen3TTSModel

os.chdir(f"{WORKSPACE}/qwen3-tts-hk-cantonese-finetune/Dataset-Cantonese-Training")

# 尋找最\u5f8c\u4e00\u500b checkpoint
CHECKPOINT_DIR = f"{WORKSPACE}/output_hk_cantonese_lora"
checkpoints = [d for d in os.listdir(CHECKPOINT_DIR) if d.startswith("checkpoint-")]
latest_ckpt = sorted(checkpoints, key=lambda x: int(x.split("-")[1]))[-1]
ckpt_path = f"{CHECKPOINT_DIR}/{latest_ckpt}"
print(f"\ud83d\udd0d Latest checkpoint: {ckpt_path}")

print("\ud83d\ude80 Loading model (will take ~3 min)...")
model = Qwen3TTSModel.from_pretrained(
    ckpt_path,
    device_map="cuda:0",
    torch_dtype=torch.bfloat16,
)
print("\u2705 Model loaded!")

# 測\u8a66\u7522\u751f
test_texts = [
    "\u5582\uff0c\u4f60\u98df\u4e86\u98ef\u672a\u5462\uff1f\u4eca\u665a\u60f3\u4e0d\u60f3\u53bb\u6253\u9084\u721b\uff1f",
    "\u9999\u6e2f\u6709\u5f88\u591a\u7f8e\u98df\uff0c\u4f60\u6700\u559c\u6b61\u5403\u4ec0\u9ebc\uff1f",
]

for i, text in enumerate(test_texts):
    print(f"\n\ud83d\udd0a [{i+1}] {text}")
    wavs, sr = model.generate_custom_voice(
        text=text,
        speaker=SPEAKER_NAME,
    )
    sf.write(f"/workspace/test_{i+1}.wav", wavs[0], sr)
    print(f"   \u2705 Saved to /workspace/test_{i+1}.wav ({len(wavs[0])/sr:.2f}s)")
    
print("\n\u2705 Inference test done!")

---

## Step 7: 上\u50b3\u5230 HuggingFace

In [ ]:
import os
os.chdir(f"{WORKSPACE}/qwen3-tts-hk-cantonese-finetune/Dataset-Cantonese-Training")

from huggingface_hub import HfApi, create_repo
import json

api = HfApi(token=HF_TOKEN)

# 導\u5efa上\u50b3\u540d\u7a31
print(f"\ud83d\udce4 Creating HF repo: {HF_REPO_ID}")
try:
    create_repo(HF_REPO_ID, repo_type="model", exist_ok=True, token=HF_TOKEN)
    print("\u2705 Repo created / already exists")
except Exception as e:
    print(f"\u26a0\ufe0f {e}")

# 上\u50b3 LoRA checkpoint
CHECKPOINT_DIR = f"{WORKSPACE}/output_hk_cantonese_lora"
checkpoints = [d for d in os.listdir(CHECKPOINT_DIR) if d.startswith("checkpoint-")]
latest_ckpt = sorted(checkpoints, key=lambda x: int(x.split("-")[1]))[-1]
ckpt_path = f"{CHECKPOINT_DIR}/{latest_ckpt}"

print(f"\ud83d\udce2 Uploading {ckpt_path} ...")
api.upload_folder(
    folder_path=ckpt_path,
    repo_id=HF_REPO_ID,
    repo_type="model",
)

# 上\u50b3 README
readme_content = f"""---\nlanguage:\n- yue\nlicense: apache-2.0\ntags:\n- cantonese\n- tts\n- qwen3-tts\n- lora\nbase_model: Qwen/Qwen3-TTS-12Hz-0.6B-Base\n---

# Qwen3-TTS Cantonese (HK) LoRA

Fine-tuned on Qwen3-TTS-12Hz-0.6B-Base using ~2000 Hong Kong Cantonese audio samples\nfrom [johnsonhk88/qwen3-tts-hk-cantonese-finetune](https://github.com/johnsonhk88/qwen3-tts-hk-cantonese-finetune).\n
## Usage\n```python\nfrom qwen_tts import Qwen3TTSModel\nimport torch\n\nmodel = Qwen3TTSModel.from_pretrained(\n    \"{HF_USERNAME}/qwen3-tts-hk-cantonese-lora\",\n    device_map=\"cuda:0\",\n    torch_dtype=torch.bfloat16,\n)\nwavs, sr = model.generate_custom_voice(\n    text=\"\u5582\uff0c\u4f60\u98df\u4e86\u98ef\u672a\u5462\uff1f\",\n    speaker=\"hk_cantonese_speaker\"\n)\n```\n\n## Training Details\n- Epochs: {NUM_EPOCHS}\n- Batch size: {BATCH_SIZE}\n- Learning rate: {LEARNING_RATE}\n- LoRA rank: {LORA_RANK}, alpha: {LORA_ALPHA}\n"""

with open("/workspace/README.md", "w") as f:
    f.write(readme_content)

api.upload_file(
    path_or_fileobj="/workspace/README.md",
    path_in_repo="README.md",
    repo_id=HF_REPO_ID,
    repo_type="model",
)

print(f"\n\u2705\u2705\u2705 All done!\n\ud83c\udf10 huggingface.co/{HF_REPO_ID}")

---

## \u26a0\ufe0f 假如中\u9014\u65b7\u7d93\u600e\u9ebc\u8fa6？

如\u679c Colab \u65b7\u7dda\u4e86\uff0c\u91cd\u65b0\u6253\u958b\u6b64\u7b46\u8a18\u672c\uff0c\u5728 **Step 3** \u7684\u8106\u7387\u4e0b\u8f09\u5b8c\u6210\u5f8c\uff0c\u76f4\u63a5\u4ece **Step 4** \u958b\u59cb\u6e2c\u8a66\u4e0d\u540c\u7684 checkpoint\u3002